# L84 · LoRA 低秩适配 — W = W₀ + BA，用 0.1% 参数量精调 GPT-style 模型

**学习目标**：实现 `LoRALinear` 层，理解为什么低秩分解（Low-Rank Decomposition）能把可训练参数（Trainable Parameters）减少 97–99%，做到「只动少数参数却保留预训练知识」。

🔗 **Aurora 连接**：`LoRALinear` 将插入 Aurora 的 `src/aurora/llm/`（大模型微调）和 `src/aurora/speech/`（语音微调）。当前 `src/aurora/llm/` 已有 `kvcache.py`（L85 主题）、`retrieve.py`、`sample.py`；`finetune.py` 为计划中模块（planned）——语音微调功能将归入 `aurora.speech`（`aurora.asr` 子包不存在，勿引用）。


← **上一课**　[L83 · Transformer 从零复现](L83_transformer.ipynb)

> 上节课学习了 **Transformer 从零复现**：多头注意力 + 位置编码 + Feed-Forward 完整实现。  
> 本课将探讨 **LoRA 低秩适配**。

全参数微调（Full Parameter Fine-Tuning）一个 7B 模型需要存储 7B 个梯度和优化器状态，显存（Video RAM，VRAM）需求往往是模型本身的 3–4 倍。LoRA 的洞察是：预训练权重（Pre-trained Weights）的更新量 `ΔW` 天然是低秩的——用两个小矩阵 `B @ A` 来近似 `ΔW`，只训练这两个矩阵，推理时可直接合并回原权重，推理成本为零。

In [ ]:
import torch
import torch.nn as nn

## 1. LoRA 核心公式

原始线性层的前向传播：`h = W_pretrained @ x`

加入 LoRA 后：`h = (W_pretrained + B @ A) @ x`

其中：
- `W_pretrained` 形状 `(d_out, d_in)`，**冻结（Freeze），不参与梯度**
- `A` 形状 `(r, d_in)`，`r << min(d_in, d_out)`
- `B` 形状 `(d_out, r)`
- 可学习参数只有 `A` 和 `B`，合计 `r * (d_in + d_out)` 个

In [ ]:
# 直观演示：低秩矩阵的秩上界
d_in, d_out, r = 512, 512, 8
A = torch.randn(r, d_in)
B = torch.zeros(d_out, r)
delta_W = B @ A
print(f'delta_W 形状: {delta_W.shape}')
print(f'delta_W 的秩 (理论上 <= r={r}): {torch.linalg.matrix_rank(delta_W)}')

# 非零 B 时验证秩
B2 = torch.randn(d_out, r)
delta_W2 = B2 @ A
print(f'随机 B 时 delta_W 的秩: {torch.linalg.matrix_rank(delta_W2)} (<=r={r})')

## 2. 初始化策略：B=0，A~N(0, 0.01)

**为什么 B 初始化为零？** 训练开始时 `B @ A = 0`，即 `ΔW = 0`，模型输出与原始预训练权重完全相同。这保证了微调的「无扰动启动」——若随机初始化 B，第一步的输出会有随机偏差，破坏预训练知识。

**为什么 A 用随机高斯？** 需要打破对称性，让 `A` 的各行探索不同方向；否则所有行相同，等效于 `r=1`。

**缩放因子 alpha**：实际输出加上 `(alpha/r) * B @ A @ x`，`alpha` 是超参，常取与 `r` 相同的值（等效缩放为 1），也可调大来增强 LoRA 的影响力。

In [ ]:
# 演示初始化后 delta_W 确实为零
r, d = 8, 64
lora_A = nn.Parameter(torch.randn(r, d) * 0.01)
lora_B = nn.Parameter(torch.zeros(d, r))
delta = lora_B @ lora_A
print(f'初始化后 delta_W 所有元素为零: {torch.allclose(delta, torch.zeros_like(delta))} ✅')

# 一步梯度更新后 B 不再为零
x_demo = torch.randn(4, d)
out = x_demo @ lora_A.T @ lora_B.T
loss = out.sum()
loss.backward()
print(f'B 的梯度非零 (|grad|={lora_B.grad.abs().mean():.4f})，更新后 LoRA 有贡献 ✅')

## 3. 参数效率：降低 97% 可训练参数

以 `d=512, r=8` 为例：

```
原始权重  W: 512 × 512 = 262,144 参数  （冻结）
LoRA A:      8 × 512 =   4,096 参数  （可训练）
LoRA B:    512 × 8   =   4,096 参数  （可训练）
合计可训练:              8,192 参数
节省比例: 1 - 8192/262144 = 96.9%
```

实际 LLM 有数百个线性层，每层都用 LoRA 替换后，整体可训练参数通常只占模型总参数的 0.1%–1%，显存和计算量大幅下降。

In [ ]:
# 参数量对比
def count_params(params_iter):
    return sum(p.numel() for p in params_iter)

d = 512
base = nn.Linear(d, d)
total = count_params(base.parameters())
print(f'全量线性层 参数: {total:,}')

for r in [4, 8, 16, 32]:
    lora_params = r * d + d * r  # A + B
    saving = 1 - lora_params / total
    print(f'LoRA r={r:2d}: 可训练 {lora_params:6,}  节省 {saving:.1%}')

## 4. ✏️ 实现 `class LoRALinear(nn.Module)`

**推理路线**：
1. `__init__` 中创建 `self.base = nn.Linear(in_features, out_features)`，   然后 `self.base.requires_grad_(False)` 冻结所有 base 参数
2. `self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)`
3. `self.lora_B = nn.Parameter(torch.zeros(out_features, rank))`
4. `self.scale = alpha / rank`（控制 LoRA 贡献的幅度）
5. `forward(x)`：`self.base(x) + (x @ self.lora_A.T) @ self.lora_B.T * self.scale`
   ——注意：`nn.Linear` 的权重约定是 `(out, in)`，乘输入时需要 `.T`；   这里直接右乘 `A.T` 和 `B.T` 对任意 batch 形状都成立

**参考输入输出**：
```
l = LoRALinear(512, 512, rank=8, alpha=8.0)
可训练参数 = 2 * 512 * 8 = 8,192
非可训练（base W+b）= 512*512 + 512 = 262,656
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=1.0):
        super().__init__()
        # ✏️ TODO: 创建 self.base (nn.Linear) 并冻结
        # ✏️ TODO: 创建 self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        # ✏️ TODO: 创建 self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        # ✏️ TODO: self.scale = alpha / rank
        raise NotImplementedError

    def forward(self, x):
        # ✏️ TODO: base_out = self.base(x)
        # ✏️ TODO: lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        # ✏️ TODO: return base_out + self.scale * lora_out
        raise NotImplementedError

In [ ]:
try:
    # 检查 1：可训练参数数量
    l = LoRALinear(64, 64, rank=4)
    trainable = sum(p.numel() for p in l.parameters() if p.requires_grad)
    expected = 2 * 64 * 4
    assert trainable == expected, f'期望 {expected}，得到 {trainable}'
    print(f'✅ 可训练参数: {trainable} == 2×64×4')

    # 检查 2：前向输出形状正确
    x = torch.randn(3, 64)   # batch=3
    y = l(x)
    assert y.shape == (3, 64), f'输出形状错误: {y.shape}'
    print(f'✅ 输出形状: {y.shape}')

    # 检查 3：初始化时 LoRA 增量为零（B=0 → lora_out=0）
    lora_out = (x @ l.lora_A.T) @ l.lora_B.T
    assert torch.allclose(lora_out, torch.zeros_like(lora_out))
    print('✅ 初始化时 LoRA 增量为零，不扰动预训练输出')

    # 检查 4：三维输入（batch, seq, d）也能正确处理
    x3d = torch.randn(2, 5, 64)
    y3d = l(x3d)
    assert y3d.shape == (2, 5, 64)
    print(f'✅ 三维输入形状: {x3d.shape} → 输出 {y3d.shape}')
    # 检查 5: scale factor 正确 (alpha / rank)
    l_scale = LoRALinear(64, 64, rank=4, alpha=8.0)
    assert abs(l_scale.scale - 2.0) < 1e-6, f'scale 期望 2.0，得到 {l_scale.scale}'
    print(f'\u2705 scale = alpha/rank = 8.0/4 = {l_scale.scale}')

except NotImplementedError:
    print('⬜ 未实现')

## 5. 参数实验：rank 对微调效果的影响

下面模拟一个极简的「关键词识别」任务：用 LoRA 微调一个 d→d→1 网络，对比不同设置。

以 `d=64` 为例（实验实际维度）：

| 设置 | LoRA 可训练参数 | 占全参数 d×d 的比例 | 预期现象 |
|------|----------------|---------------------|----------|
| 全参数微调 | d²+d ≈ 4,160 | 100% | 收敛最快，显存最大 |
| LoRA `rank=4` | 2×d×4 = 512 | ~12.5% | 参数极少，轻微欠拟合 |
| LoRA `rank=16` | 2×d×16 = 2,048 | ~50% | 比 rank=4 多 4 倍参数，表达力更强 |

> **注意**：教材中常见的“~6%”是针对 d=512、rank=16 的情形（2×512×16 / 512² ≈ 6.25%）。
> 本实验使用 d=64 以加速演示，此时 rank=16 约占 50%，体现的是相同 *比例关系*，而非工业级绝对数字。

**预期现象**：步数相同时准确率全参数 ≥ rank=16 > rank=4；三者差距通常 < 2%，说明 LoRA 在大幅压缩参数下仍能保留绝大部分拟合能力。


In [ ]:
torch.manual_seed(42)
d = 64  # 特征维度
steps = 300
lr = 1e-2

def run_experiment(label, model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        x = torch.randn(64, d)
        y = (x[:, 0] > 0).float()   # 规则：第一维正 → 类别 1
        pred = model(x).squeeze(-1)
        loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x = torch.randn(1024, d)
        y = (x[:, 0] > 0).float()
        pred = model(x).squeeze(-1)
        acc = ((pred > 0) == y.bool()).float().mean().item()
    print(f'{label:25s}  可训练参数: {trainable:6,}  准确率: {acc:.3f}')

try:
    # 全参数：d→d（全量可训练）+ d→1，与 LoRA 变体架构深度一致
    full = nn.Sequential(nn.Linear(d, d), nn.Linear(d, 1))
    run_experiment('全参数微调', full)

    # LoRA rank=4：冻结 base d→d，仅训练低秩部分 + 最终 d→1
    lora4 = nn.Sequential(LoRALinear(d, d, rank=4, alpha=4.0), nn.Linear(d, 1))
    run_experiment('LoRA rank=4', lora4)

    # LoRA rank=16
    lora16 = nn.Sequential(LoRALinear(d, d, rank=16, alpha=16.0), nn.Linear(d, 1))
    run_experiment('LoRA rank=16', lora16)
except NotImplementedError:
    print('□ 请先完成 LoRALinear 练习（cell 10）再运行本实验')


## 本课收束

本节实现了 `LoRALinear`，它在冻结的 `base` 线性层旁并联低秩矩阵 `lora_B @ lora_A`，`forward` 输出两者之和，可训练参数仅占原层的 3%–6%，且初始化保证了无扰动启动。该模块将来将插入 Aurora 的 `src/aurora/llm/finetune.py`（计划中）和 `src/aurora/speech/`（计划中），替换 Transformer 中的每个 `nn.Linear`；目前 `src/aurora/llm/` 中已有 `kvcache.py` / `retrieve.py` / `sample.py`，`aurora.asr` 子包尚未建立，语音微调功能归入 `aurora.speech`。下一节（L85）将从零实现 KV-Cache，把自注意力的键值矩阵缓存起来，将推理时间复杂度从 O(seq²) 降到 O(seq)。


## ✏️ 闭卷推导检查格 — LoRA 参数量对比

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：原始权重矩阵 $W \in \mathbb{R}^{d_{out} \times d_{in}}$，LoRA 注入 $\Delta W = BA$，其中 $B \in \mathbb{R}^{d_{out} \times r}$，$A \in \mathbb{R}^{r \times d_{in}}$。

1. 原始矩阵参数量：$|W| = ?$
2. LoRA 参数量：$|A| + |B| = ?$
3. 参数节省比例（LoRA / 原始）= ?
4. 若 $d_{in} = d_{out} = 768$，$r = 8$，计算节省比例（保留两位小数）

（在此处写推导...）

In [ ]:
# 验证：参数量计算
def lora_savings(d_in, d_out, r):
    original = d_in * d_out
    lora = r * (d_in + d_out)
    ratio = lora / original
    return original, lora, ratio

cases = [(768, 768, 8), (1024, 1024, 16), (768, 3072, 4)]
for d_in, d_out, r in cases:
    orig, lo, ratio = lora_savings(d_in, d_out, r)
    print(f"d_in={d_in} d_out={d_out} r={r}: "
          f"orig={orig:,} lora={lo:,} ratio={ratio:.2%}")

# 具体验证 768×768 r=8
orig, lo, ratio = lora_savings(768, 768, 8)
assert abs(ratio - 8*(768+768)/(768*768)) < 1e-9
expected = 2*8*768 / (768*768)   # ≈ 2.08%
assert abs(ratio - expected) < 1e-6, f"ratio={ratio:.4f} vs expected={expected:.4f}"
print(f"\n✅ LoRA 节省比例验证：768×768 r=8 → 使用 {ratio:.2%} 参数")

---

→ **下一课**　[L85 · KV-Cache 从零实现](L85_kv_cache.ipynb)

> 下节课将学习 **KV-Cache 从零实现**：键值缓存矩阵更新，O(seq²)→O(seq) 加速。